In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
df = pd.read_csv('data/Cleaned_loan_records.csv')

In [3]:
X = df.drop('Status',axis=1)
y = df.Status

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42,stratify=y)

In [5]:
X_train

,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,loan_amount,rate_of_interest,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,dtir1,has_Upfront_charges
23068,ncf,Male,nopre,type3,p3,l1,nopc,nob/c,916500,4.30960,...,CRIF,817,EXP,65-74,to_inst,79.145078,North,direct,30.0,0
235,ncf,Male,nopre,type1,p4,l1,nopc,nob/c,566500,3.62500,...,CRIF,693,CIB,35-44,to_inst,61.045259,North,direct,23.0,0
95591,ncf,Female,nopre,type1,p3,l1,nopc,nob/c,296500,3.99000,...,CIB,698,CIB,55-64,to_inst,46.473354,North,direct,22.0,1
42538,cf,Male,nopre,type1,p4,l1,nopc,nob/c,156500,3.87500,...,CIB,528,EXP,>74,to_inst,56.294964,central,direct,20.0,1
144762,cf,Male,nopre,type1,p4,l1,nopc,nob/c,286500,4.12500,...,CRIF,632,CIB,35-44,not_inst,93.019481,central,direct,47.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
102388,cf,Joint,nopre,type1,p1,l1,nopc,nob/c,656500,4.50000,...,EXP,695,EXP,45-54,not_inst,74.772210,south,direct,34.0,0
18618,cf,Joint,nopre,type1,p4,l1,nopc,nob/c,616500,4.01645,...,CRIF,785,EXP,25-34,to_inst,83.536585,North,direct,47.0,0
49626,cf,Sex Not Available,nopre,type1,p4,l1,nopc,nob/c,246500,3.62500,...,CIB,587,EXP,35-44,to_inst,57.593458,south,direct,39.0,1
27609,cf,Joint,pre,type1,p4,l1,nopc,nob/c,386500,4.25000,...,EXP,711,EXP,>74,not_inst,39.119433,North,direct,49.0,1


In [6]:
from sklearn.pipeline import Pipeline
from src.transformers.Transformers import EnCoder,LogTransform

In [7]:
numcols = X_train.select_dtypes(exclude=object).columns

In [8]:
ageLabelOrder  = ['<25','25-34', '35-44', '45-54','55-64', '65-74', '>74']
creditypeLabelOrder = [ 'CIB', 'CRIF' ,'EXP','EQUI']
totalunitLabelorder = ['1U','2U','3U','4U']
ordinalcols = ['age','total_units','credit_type']
ordinalCategories = [ageLabelOrder,totalunitLabelorder,creditypeLabelOrder]

In [9]:
ohecols = [
    "Gender",
    "loan_limit",
    "approv_in_adv",
    "Credit_Worthiness",
    "loan_type",
    "loan_purpose",
    "open_credit",
    "business_or_commercial",
    "Neg_ammortization",
    "interest_only",
    "lump_sum_payment",
    "construction_type",
    "occupancy_type",
    "Secured_by",
    "submission_of_application",
    "Region",
    "Security_Type",
    "co-applicant_credit_type"
]

In [10]:
lgr_pipe = Pipeline([
    ('encoder',EnCoder(OrdinalCategories=ordinalCategories,OrdinalCols=ordinalcols,OheCols=ohecols)),
    ('log1p',LogTransform(cols=numcols)),
    ('lgr',LogisticRegression(class_weight='balanced',max_iter=1000))
])

In [11]:
lgr_pipe.fit(X_train,y_train)

Pipeline(steps=[('encoder',
                 EnCoder(OheCols=['Gender', 'loan_limit', 'approv_in_adv',
                                  'Credit_Worthiness', 'loan_type',
                                  'loan_purpose', 'open_credit',
                                  'business_or_commercial', 'Neg_ammortization',
                                  'interest_only', 'lump_sum_payment',
                                  'construction_type', 'occupancy_type',
                                  'Secured_by', 'submission_of_application',
                                  'Region', 'Security_Type',
                                  'co-applicant_credit_typ...
                                            ['1U', '2U', '3U', '4U'],
                                            ['CIB', 'CRIF', 'EXP', 'EQUI']],
                         OrdinalCols=['age', 'total_units', 'credit_type'])),
                ('log1p',
                 LogTransform(cols=Index(['loan_amount', 'rate_of_interest', 'Upfront_charges', 'term',
       'property_value', 'income', 'Credit_Score', 'LTV', 'dtir1',
       'has_Upfront_charges'],
      dtype='object'))),
                ('lgr',
                 LogisticRegression(class_weight='balanced', max_iter=1000))])

In [12]:
lgr_pipe.score(X_test,y_test)

0.910746513535685

In [13]:
from sklearn.metrics import classification_report,confusion_matrix
from sklearn.model_selection import cross_val_score

In [14]:
def ReportCard(n,m,X_train, X_test, y_train, y_test,cv=7):
    m.fit(X_train,y_train)
    y_pred = m.predict(X_test)
    print(n)
    print('-'*30)
    print(confusion_matrix(y_true=y_test,y_pred=y_pred))
    print('-'*30)
    print(classification_report(y_true=y_test,y_pred=y_pred))
    print('-'*30)
    X = pd.concat([X_train,X_test],axis=0)
    y =pd.concat([y_train,y_test],axis=0)
    cvs = cross_val_score(m,X,y,scoring='recall',cv=cv)
    print(f"cvs:\n\tmean: {cvs.mean()} | std: {cvs.std()} | min: {cvs.min()} | max: {cvs.max()}")

In [15]:
ReportCard('LogisticRegression',lgr_pipe,X_train, X_test, y_train, y_test)

LogisticRegression
------------------------------
[[33325  3645]
 [  707 11083]]
------------------------------
              precision    recall  f1-score   support

           0       0.98      0.90      0.94     36970
           1       0.75      0.94      0.84     11790

    accuracy                           0.91     48760
   macro avg       0.87      0.92      0.89     48760
weighted avg       0.92      0.91      0.91     48760

------------------------------
cvs:
	mean: 0.9407451297275234 | std: 0.0024837681465524762 | min: 0.9373040752351097 | max: 0.9439655172413793


In [16]:
rf_pipe = Pipeline([
    ('encoder',EnCoder(OrdinalCategories=ordinalCategories,OrdinalCols=ordinalcols,OheCols=ohecols)),
    ('log1p',LogTransform(cols=numcols)),
    ('rf',RandomForestClassifier(random_state=42,n_estimators=250,class_weight='balanced',n_jobs=-1))
])

In [17]:
ReportCard('RF',rf_pipe,X_train, X_test, y_train, y_test)

RF
------------------------------
[[36152   818]
 [ 1077 10713]]
------------------------------
              precision    recall  f1-score   support

           0       0.97      0.98      0.97     36970
           1       0.93      0.91      0.92     11790

    accuracy                           0.96     48760
   macro avg       0.95      0.94      0.95     48760
weighted avg       0.96      0.96      0.96     48760

------------------------------
cvs:
	mean: 0.9132587846212186 | std: 0.0027052540130303756 | min: 0.9092868338557993 | max: 0.917319749216301


In [18]:
from sklearn.metrics import roc_auc_score

In [19]:
y_prob = rf_pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_prob)


np.float64(0.9931903891081024)

In [20]:
y_prob = lgr_pipe.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_prob)


np.float64(0.9800256448905342)

In [21]:
rf_pipe2 = Pipeline([
    ('encoder',EnCoder(OrdinalCategories=ordinalCategories,OrdinalCols=ordinalcols,OheCols=ohecols)),
    ('log1p',LogTransform(cols=numcols)),
    ('rf',RandomForestClassifier(
    n_estimators=400,
    max_depth=6,
    min_samples_leaf=4,
    max_features="log2",
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
))
]) 

In [22]:
ReportCard('RF 2',rf_pipe2,X_train, X_test, y_train, y_test)

RF 2
------------------------------
[[31634  5336]
 [  194 11596]]
------------------------------
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     36970
           1       0.68      0.98      0.81     11790

    accuracy                           0.89     48760
   macro avg       0.84      0.92      0.86     48760
weighted avg       0.92      0.89      0.89     48760

------------------------------
cvs:
	mean: 0.9863408281124028 | std: 0.0008874455437625022 | min: 0.9851067999216148 | max: 0.9876567398119123


In [23]:
y_prob = rf_pipe2.predict_proba(X_test)[:,1]
roc_auc_score(y_test, y_prob)


np.float64(0.9828388306498884)

In [24]:
from joblib import dump

In [25]:
dump(rf_pipe,'models/rf/LoanDefaulterPredictor_100326_v1.joblib')

['models/rf/LoanDefaulterPredictor_100326_v1.joblib']